In [25]:
import json, math, re, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

sys.path.insert(0, "kev")
from models.kev_pointer import KevPointerModel


In [26]:
splits_by_year = json.loads(Path("splits/split.json").read_text())

frames = []
for year, split_to_vars in splits_by_year.items():
    df = pd.read_parquet(f"data/issp{year}/issp{year}_US_long.parquet")
    for col in ("option_codes", "option_texts"):
        df[col] = df[col].map(list)
    var_to_split = {
        var_code: split_name
        for split_name, var_codes in split_to_vars.items()
        for var_code in var_codes
    }
    df["year"] = year
    df["split"] = [var_to_split[v] for v in df["v"]]
    df["qid"] = df["year"] + ":" + df["v"]
    frames.append(df)

all_responses_df = pd.concat(frames, ignore_index=True)
test_df = all_responses_df[all_responses_df["split"] == "test"].reset_index(drop=True)
test_df.head(2)

,CASEID,c_alphan,SEX,AGE,EDULEVEL,RELIGGRP,ATTEND,PARTY_LR,VOTE_LE,URBRURAL,...,question,type,reversed,option_codes,option_texts,answer_code,answer_text,year,split,qid
0,2015008400000003,US,Male,72,"Lower level tertiary, first stage (also techni...",Catholic,Once a week,"Right, conservative",Yes,The suburbs or outskirts of a big city,...,Please show how important you personally think...,score,True,"[1, 2, 3, 4, 5]","[Very important, Important, Neither important ...",2,Important,2015,test,2015:v11
1,2015008400000005,US,Female,55,"Upper level tertiary (Master, Doctor)",No religion,Never,"Left, center left",Yes,The suburbs or outskirts of a big city,...,Please show how important you personally think...,score,True,"[1, 2, 3, 4, 5]","[Very important, Important, Neither important ...",2,Important,2015,test,2015:v11


In [28]:
persona_cols = [
    col for col in all_responses_df.columns[1:all_responses_df.columns.get_loc("v")]
    if col != "CASEID"
]
country = all_responses_df["c_alphan"].iloc[0]

PERSONA_NAMES = {
    "c_alphan": "Country",
    "SEX": "Sex",
    "AGE": "Age",
    "EDULEVEL": "Education",
    "RELIGGRP": "Religion",
    "ATTEND": "Attendance of religious services",
    "PARTY_LR": "Political ideology",
    "VOTE_LE": "Voted in the last general election",
    "URBRURAL": "Place of living",
    "MARITAL": "Marital status",
    "MAINSTAT": "Main activity",
    "UNION": "Trade union membership",
    f"{country}_PRTY": "Political party",
    f"{country}_INC": "Income",
    f"{country}_REG": "Region",
    f"{country}_ETHN1": "Ancestry",
}
if set(persona_cols) != set(PERSONA_NAMES.keys()):
    raise ValueError(f"Persona columns mismatch: {set(persona_cols) ^ set(PERSONA_NAMES.keys())}")

def _map_us_income(val):
    nums = [int(n.replace(".", "").replace(" ", "")) for n in re.findall(r"\d+(?:[.\s]\d{3})*", val)]
    if len(nums) == 1 and val.startswith("Under ") and nums[0] <= 30_000:
        return "Less than $30,000"
    if len(nums) == 1 and "or more" in val and nums[0] >= 100_000:
        return "$100,000 or more"
    if len(nums) == 2 and nums[0] < nums[1]:
        low, high = nums
        if high < 30_000:
            return "Less than $30,000"
        if low >= 100_000:
            return "$100,000 or more"
        if 30_000 <= low and high < 110_000:
            return val
    raise ValueError(f"Unexpected US_INC value: {val!r}")

PERSONA_VALUE_MAP = {
    "c_alphan": {"US": "United States"},
    "SEX": {"Male": "Male", "Female": "Female"},
    "EDULEVEL": {
        "No formal education": "Less than high school",
        "No (formal) education, incomplete primary": "Less than high school",
        "No education, incomplete primary": "Less than high school",
        "Primary school (elementary education)": "Less than high school",
        "Primary education": "Less than high school",
        "Primary school": "Less than high school",
        "Lower secondary (secondary completed does not allow entry to university: obligatory school)": "Less than high school",
        "Lower secondary (secondary completed that does not allow entry to university: end of obligatory school)": "Less than high school",
        "Lower secondary education": "Less than high school",
        "Lower secondary": "Less than high school",
        "Upper secondary (programs that allows entry to university)": "Upper secondary (programs that allows entry to university)",
        "Upper secondary (programs that allow entry to university)": "Upper secondary (programs that allow entry to university)",
        "Upper secondary education": "Upper secondary education",
        "Upper secondary": "Upper secondary",
        "Post secondary, non-tertiary (other upper secondary programs toward labour market or technical formation)": "Post secondary, non-tertiary (other upper secondary programs toward labour market or technical formation)",
        "Post secondary, non-tertiary (other upper secondary programs toward the labour market or technical formation)": "Post secondary, non-tertiary (other upper secondary programs toward the labour market or technical formation)",
        "Post secondary, non-tertiary": "Post secondary, non-tertiary",
        "Short-cycle tertiary": "Short-cycle tertiary",
        "Lower level tertiary, first stage (also technical schools at a tertiary level)": "College graduate/some postgrad",
        "Lower level tertiary, BA": "College graduate/some postgrad",
        "Lower tertiary, BA": "College graduate/some postgrad",
        "Upper level tertiary (Master, Doctor)": "College graduate/some postgrad",
        "Upper level tertiary, MA": "College graduate/some postgrad",
        "Upper tertiary, MA": "College graduate/some postgrad",
        "PhD, Post tertiary specialization": "College graduate/some postgrad",
        "PhD, Post Tertiary Specialization": "College graduate/some postgrad",
    },
    "RELIGGRP": {
        "No religion": "Atheist",
        "Islamic": "Muslim",
        "Protestant": "Protestant",
        "Catholic": "Catholic",
        "Orthodox": "Orthodox",
        "Other Christian": "Other Christian",
        "Jewish": "Jewish",
        "Buddhist": "Buddhist",
        "Hindu": "Hindu",
        "Other Asian religions": "Other Asian religions",
        "Other Asian Religions": "Other Asian Religions",
        "Other religions": "Other religions",
        "Other Religions": "Other Religions",
    },
    "PARTY_LR": {
        "Left, center left": "Liberal",
        "Left / center left": "Liberal",
        "Center, liberal": "Liberal",
        "Center / liberal": "Liberal",
        "Right, conservative": "Conservative",
        "Right / conservative": "Conservative",
        "Other": "Other",
    },
    f"{country}_PRTY": {
        "Democrat (Obama)": "Democrat",
        "Democrat (Clinton)": "Democrat",
        "Democrat": "Democrat",
        "Republican (Romney)": "Republican",
        "Republican (Trump)": "Republican",
        "Republican": "Republican",
        "Other Candidate": "Other Candidate",
        "Other candidate": "Other candidate",
    },
    f"{country}_INC": {
        val: _map_us_income(val)
        for val in all_responses_df[f"{country}_INC"].dropna().unique()
    },
    f"{country}_REG": {
        "New England": "Northeast",
        "Middle Atlantic": "Northeast",
        "South Atlantic": "South",
        "East South Central": "South",
        "West South Central": "South",
        "East North Central": "East North Central",
        "West North Central": "West North Central",
        "Mountain": "Mountain",
        "Pacific": "Pacific",
    },
}

for col, mapping in PERSONA_VALUE_MAP.items():
    unmapped = set(all_responses_df[col].dropna().unique()) - set(mapping.keys())
    if unmapped:
        raise KeyError(f"Unmapped values in {col}: {sorted(unmapped)}")

question_groups = {qid: q_df for qid, q_df in test_df.groupby("qid", sort=False)}

print(f"Test set: {len(test_df):,} answers across {len(question_groups)} questions")

KeyError: "Unmapped values in RELIGGRP: ['Other Christian, NZ,PL: Christian', 'Other Christian, NZ: Christian']"

In [30]:
test_df['RELIGGRP'].value_counts()

RELIGGRP
Protestant                           40349
No religion                          23888
Catholic                             19501
Other Religions                       1970
Jewish                                1640
Other Christian                       1624
Buddhist                               771
Islamic                                689
Hindu                                  667
Orthodox                               558
Other Christian, NZ: Christian         258
Other religions                        114
Other Asian Religions                   74
Other Christian, NZ,PL: Christian       59
Other Asian religions                    5
Name: count, dtype: int64

In [4]:
sample_question_df = question_groups['2015:v11']
sample_question_df

,CASEID,c_alphan,SEX,AGE,EDULEVEL,RELIGGRP,ATTEND,PARTY_LR,VOTE_LE,URBRURAL,...,question,type,reversed,option_codes,option_texts,answer_code,answer_text,year,split,qid
0,2015008400000003,US,Male,72,"Lower level tertiary, first stage (also techni...",Catholic,Once a week,"Right, conservative",Yes,The suburbs or outskirts of a big city,...,Please show how important you personally think...,score,True,"[1, 2, 3, 4, 5]","[Very important, Important, Neither important ...",2,Important,2015,test,2015:v11
1,2015008400000005,US,Female,55,"Upper level tertiary (Master, Doctor)",No religion,Never,"Left, center left",Yes,The suburbs or outskirts of a big city,...,Please show how important you personally think...,score,True,"[1, 2, 3, 4, 5]","[Very important, Important, Neither important ...",2,Important,2015,test,2015:v11
2,2015008400000008,US,Female,23,Upper secondary (programs that allows entry to...,Catholic,"2 or 3 times a month, nearly every week",NaN,NaN,A big city,...,Please show how important you personally think...,score,True,"[1, 2, 3, 4, 5]","[Very important, Important, Neither important ...",1,Very important,2015,test,2015:v11
3,2015008400000010,US,Male,71,"Post secondary, non-tertiary (other upper seco...",No religion,Never,"Right, conservative",Yes,A big city,...,Please show how important you personally think...,score,True,"[1, 2, 3, 4, 5]","[Very important, Important, Neither important ...",2,Important,2015,test,2015:v11
4,2015008400000011,US,Female,33,Upper secondary (programs that allows entry to...,Protestant,"2 or 3 times a month, nearly every week",NaN,No,A big city,...,Please show how important you personally think...,score,True,"[1, 2, 3, 4, 5]","[Very important, Important, Neither important ...",1,Very important,2015,test,2015:v11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1463,2015008400002858,US,Female,41,Upper secondary (programs that allows entry to...,Protestant,Once a month,"Left, center left",Yes,A farm or home in the country,...,Please show how important you personally think...,score,True,"[1, 2, 3, 4, 5]","[Very important, Important, Neither important ...",2,Important,2015,test,2015:v11
1464,2015008400002860,US,Male,57,Upper secondary (programs that allows entry to...,Protestant,about once or twice a year,NaN,No,A farm or home in the country,...,Please show how important you personally think...,score,True,"[1, 2, 3, 4, 5]","[Very important, Important, Neither important ...",4,Not important,2015,test,2015:v11
1465,2015008400002862,US,Female,75,Upper secondary (programs that allows entry to...,Protestant,Once a week,NaN,Yes,A farm or home in the country,...,Please show how important you personally think...,score,True,"[1, 2, 3, 4, 5]","[Very important, Important, Neither important ...",4,Not important,2015,test,2015:v11
1466,2015008400002865,US,Female,87,Upper secondary (programs that allows entry to...,Protestant,Once a week,NaN,Yes,A farm or home in the country,...,Please show how important you personally think...,score,True,"[1, 2, 3, 4, 5]","[Very important, Important, Neither important ...",5,Not important at all,2015,test,2015:v11


In [6]:
def format_persona_value(col, val):
    if col not in PERSONA_NAMES:
        raise KeyError(f"Unexpected persona column: {col!r}")
    if col in PERSONA_VALUE_MAP:
        return PERSONA_VALUE_MAP[col][val]
    return val

def build_persona(respondent):
    return "\n".join(
        f"{PERSONA_NAMES[col]}: {format_persona_value(col, respondent[col])}"
        for col in persona_cols
        if pd.notna(respondent[col])
    )

def build_record(respondent):
    qid = respondent["qid"]
    q_type = respondent["type"]
    option_codes = respondent["option_codes"]
    option_texts = respondent["option_texts"]
    answer_code = respondent["answer_code"]
    if len(option_codes) != len(option_texts):
        raise ValueError(f"{qid}: option_codes ({len(option_codes)}) != option_texts ({len(option_texts)})")
    if answer_code not in option_codes:
        raise ValueError(f"{qid}: answer_code {answer_code!r} not in option_codes {option_codes!r}")
    one_hot_target = [1.0 if code == answer_code else 0.0 for code in option_codes]
    if q_type == "score":
        criteria = list(option_texts)
        target = {str(idx): prob for idx, prob in enumerate(one_hot_target)}
    elif q_type == "choice":
        criteria = {text: None for text in option_texts}
        target = dict(zip(option_texts, one_hot_target))
    else:
        raise ValueError(f"{qid}: unexpected question type {q_type!r}")
    return {
        "state": build_persona(respondent),
        "n": {qid: 1},
        "questions": {
            qid: {
                "type": q_type,
                "instructions": respondent["question"],
                "criteria": criteria,
                "target": target,
            }
        },
    }

def build_question_records(q_df):
    return [build_record(row) for _, row in q_df.iterrows()]


In [8]:
@torch.no_grad()
def predict_probs(model, records, batch_size=64, log=False, progress=None):
    model.eval()
    batch_preds = []
    norm_fn = torch.log_softmax if log else torch.softmax
    for start in range(0, len(records), batch_size):
        batch_records = records[start:start + batch_size]
        prepared_batch = [model.prepare(record) for record in batch_records]
        logits_per_record = model.forward([encoded for encoded, _ in prepared_batch])
        for question_logits in logits_per_record:
            if len(question_logits) != 1:
                raise ValueError(f"Expected 1 question per record, got {len(question_logits)}")
            batch_preds.append(norm_fn(question_logits[0].float(), -1).cpu().numpy())
        if progress is not None:
            progress.update(1)
    return np.stack(batch_preds)

def consensus(probs):
    num_options = len(probs)
    if num_options <= 1:
        raise ValueError(f"consensus requires at least 2 options, got {num_options}")
    scale_points = np.arange(1, num_options + 1, dtype=float)
    mean_score = float(np.dot(scale_points, probs))
    return float(
        1.0 + np.sum(probs * np.log2(1.0 - np.abs(scale_points - mean_score) / (num_options - 1) + 1e-12))
    )

def evaluate_question(q_df, pred_log_probs):
    question_meta = q_df.iloc[0]
    qid = question_meta["qid"]
    q_type = question_meta["type"]
    option_codes = list(question_meta["option_codes"])
    expected_shape = (len(q_df), len(option_codes))
    if pred_log_probs.shape != expected_shape:
        raise ValueError(f"{qid}: expected pred_log_probs shape {expected_shape}, got {pred_log_probs.shape}")
    pred_probs = np.exp(pred_log_probs)

    true_option_idx = np.array([option_codes.index(code) for code in q_df["answer_code"]])
    resp_kl = float((-pred_log_probs[np.arange(len(q_df)), true_option_idx] / np.log(2)).mean())

    human_dist = np.bincount(true_option_idx, minlength=len(option_codes)) / len(true_option_idx)
    model_dist = pred_probs.mean(axis=0)
    nonzero_mask = human_dist > 0
    dist_kl = float(
        np.sum(human_dist[nonzero_mask] * np.log2(human_dist[nonzero_mask] / np.clip(model_dist[nonzero_mask], 1e-12, 1.0)))
    )

    if q_type == "score":
        w1 = float(np.abs(np.cumsum(human_dist)[:-1] - np.cumsum(model_dist)[:-1]).sum() / (len(option_codes) - 1))
        delta_cns = consensus(model_dist) - consensus(human_dist)
    elif q_type == "choice":
        w1 = delta_cns = np.nan
    else:
        raise ValueError(f"{qid}: unexpected question type {q_type!r}")

    return {
        "qid": qid,
        "year": question_meta["year"],
        "v": question_meta["v"],
        "type": q_type,
        "n": len(q_df),
        "resp_kl": resp_kl,
        "dist_kl": dist_kl,
        "w1": w1,
        "delta_cns": delta_cns,
    }


In [ ]:
sample_question_df.iloc[0]['option_codes']


[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

In [22]:
sample_question_df.iloc[0]['option_texts']


['Very important',
 'Important',
 'Neither important nor unimportant',
 'Not important',
 'Not important at all']

In [18]:
sample_question_df.iloc[0]['answer_code']


np.int64(2)

In [21]:
sample_question_df.iloc[0]['answer_text']


'Important'

In [14]:
# sample_question_df['qid']

In [11]:
build_question_records(sample_question_df.iloc[:1])

[{'state': "Country: United States\nSex: Male\nAge: 72\nEducation: Lower level tertiary, first stage (also technical schools at a tertiary level)\nReligion: Catholic\nAttendance of religious services: Once a week\nVoted party's left-right position: Right, conservative\nVoted in the last general election: Yes\nPlace of living: The suburbs or outskirts of a big city\nMarital status: Married\nMain activity: Retired\nTrade union membership: No, never\nParty voted for in the last general election: Republican (Romney)\nHousehold income: 75.000-89.999 USD\nRegion: New England\nAncestry: Italy",
  'n': {'2015:v11': 1},
  'questions': {'2015:v11': {'type': 'score',
    'instructions': 'Please show how important you personally think the following is in a job. How important is... … a job that involves personal contact with other people',
    'criteria': ['Very important',
     'Important',
     'Neither important nor unimportant',
     'Not important',
     'Not important at all'],
    'target': 

In [ ]:
device = "cuda"
model = KevPointerModel.load("jaredpalmer/kev-0.8b", device)
model.net.head.temperature = 1.0
print(model.describe())

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

{'kind': 'kev', 'base': 'Qwen/Qwen3.5-0.8B-Base', 'mode': 'loaded', 'trainable_params': 524800}


In [ ]:
batch_size = 64
total_batches = sum(math.ceil(len(q_df) / batch_size) for q_df in question_groups.values())
progress = tqdm(total=total_batches, desc="benchmark kev-0.8b", unit="batch")

rows_out = []
for qid, q_df in question_groups.items():
    records = build_question_records(q_df)
    pred_log_probs = predict_probs(model, records, batch_size=batch_size, log=True, progress=progress)
    rows_out.append(evaluate_question(q_df, pred_log_probs))
progress.close()

results_df = pd.DataFrame(rows_out)
print(results_df[["resp_kl", "dist_kl", "w1", "delta_cns"]].mean())
display(results_df.head(10))